In [2]:

import requests
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime, timedelta
from io import StringIO

# Bank of England official Statistics API
# Series code IUMBEDR = Bank Rate (official base rate)
# https://www.bankofengland.co.uk/statistics/bank-stats-technical-notes

end_date = datetime.today()
start_date = end_date - timedelta(days=365)

url = (
    "https://www.bankofengland.co.uk/boeapps/database/_iadb-FromShowColumns.asp"
    "?csv.x=yes"
    f"&Datefrom={start_date.strftime('%d/%b/%Y')}"
    f"&Dateto={end_date.strftime('%d/%b/%Y')}"
    "&SeriesCodes=IUMBEDR"
    "&CSVF=TN"
    "&UsingCodes=Y"
)

print(f"Fetching data from Bank of England Statistics API...")
print(f"Period: {start_date.strftime('%d %b %Y')} → {end_date.strftime('%d %b %Y')}\n")

response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
response.raise_for_status()

df = pd.read_csv(StringIO(response.text))
df.columns = ["Date", "Base Rate (%)"]
df["Date"] = pd.to_datetime(df["Date"], format="mixed", dayfirst=True)
df["Base Rate (%)"] = pd.to_numeric(df["Base Rate (%)"], errors="coerce")
df = df.dropna().sort_values("Date").reset_index(drop=True)

print(f"Records retrieved: {len(df)}")
print(df.to_string(index=False))

# ── Plotly chart ──────────────────────────────────────────────────────────────
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df["Date"],
    y=df["Base Rate (%)"],
    mode="lines+markers",
    line=dict(color="#C8102E", width=2.5),   # Bank of England red
    marker=dict(size=8, color="#C8102E"),
    name="Base Rate",
    hovertemplate="<b>%{x|%d %b %Y}</b><br>Base Rate: <b>%{y:.2f}%</b><extra></extra>"
))

fig.update_layout(
    title=dict(
        text="Bank of England Base Rate — Last 12 Months",
        font=dict(size=20),
        x=0.5
    ),
    xaxis=dict(title="Date", showgrid=True, gridcolor="#e5e5e5", tickformat="%b %Y"),
    yaxis=dict(title="Base Rate (%)", showgrid=True, gridcolor="#e5e5e5",
               ticksuffix="%", rangemode="tozero"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    hovermode="x unified",
    annotations=[dict(
        text='Source: <a href="https://www.bankofengland.co.uk/monetary-policy/the-bank-rate">'
             'Bank of England Statistics (IUMBEDR)</a>',
        showarrow=False, xref="paper", yref="paper",
        x=1, y=-0.16, xanchor="right", font=dict(size=11, color="grey")
    )],
    margin=dict(b=90)
)

fig.show()


Fetching data from Bank of England Statistics API...
Period: 29 May 2025 → 29 May 2026

Records retrieved: 12
      Date  Base Rate (%)
2025-05-31           4.25
2025-06-30           4.25
2025-07-31           4.25
2025-08-31           4.00
2025-09-30           4.00
2025-10-31           4.00
2025-11-30           4.00
2025-12-31           3.75
2026-01-31           3.75
2026-02-28           3.75
2026-03-31           3.75
2026-04-30           3.75


In [3]:

import requests
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime, timedelta
from io import StringIO

# Bank of England official Statistics API
# Series code IUDSOIA = SONIA (Sterling Overnight Index Average) - daily
# https://www.bankofengland.co.uk/statistics/bank-stats-technical-notes

end_date = datetime.today()
start_date = end_date - timedelta(days=365)

url_sonia = (
    "https://www.bankofengland.co.uk/boeapps/database/_iadb-FromShowColumns.asp"
    "?csv.x=yes"
    f"&Datefrom={start_date.strftime('%d/%b/%Y')}"
    f"&Dateto={end_date.strftime('%d/%b/%Y')}"
    "&SeriesCodes=IUDSOIA"
    "&CSVF=TN"
    "&UsingCodes=Y"
)

print(f"Fetching SONIA data from Bank of England Statistics API...")
print(f"Period: {start_date.strftime('%d %b %Y')} → {end_date.strftime('%d %b %Y')}\n")

resp_sonia = requests.get(url_sonia, headers={"User-Agent": "Mozilla/5.0"})
resp_sonia.raise_for_status()

df_sonia = pd.read_csv(StringIO(resp_sonia.text))
df_sonia.columns = ["Date", "SONIA Rate (%)"]
df_sonia["Date"] = pd.to_datetime(df_sonia["Date"], format="mixed", dayfirst=True)
df_sonia["SONIA Rate (%)"] = pd.to_numeric(df_sonia["SONIA Rate (%)"], errors="coerce")
df_sonia = df_sonia.dropna().sort_values("Date").reset_index(drop=True)

print(f"Records retrieved: {len(df_sonia)}")
print(df_sonia.tail(10).to_string(index=False))

# ── Plotly chart ──────────────────────────────────────────────────────────────
fig_sonia = go.Figure()

fig_sonia.add_trace(go.Scatter(
    x=df_sonia["Date"],
    y=df_sonia["SONIA Rate (%)"],
    mode="lines",
    line=dict(color="#003087", width=1.5),   # BoE navy blue
    name="SONIA",
    hovertemplate="<b>%{x|%d %b %Y}</b><br>SONIA: <b>%{y:.4f}%</b><extra></extra>"
))

fig_sonia.update_layout(
    title=dict(
        text="SONIA (Sterling Overnight Index Average) — Last 12 Months",
        font=dict(size=20),
        x=0.5
    ),
    xaxis=dict(title="Date", showgrid=True, gridcolor="#e5e5e5", tickformat="%b %Y"),
    yaxis=dict(title="Rate (%)", showgrid=True, gridcolor="#e5e5e5",
               ticksuffix="%", rangemode="tozero"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    hovermode="x unified",
    annotations=[dict(
        text='Source: <a href="https://www.bankofengland.co.uk/markets/sonia-benchmark">'
             'Bank of England Statistics (IUDSOIA)</a>',
        showarrow=False, xref="paper", yref="paper",
        x=1, y=-0.16, xanchor="right", font=dict(size=11, color="grey")
    )],
    margin=dict(b=90)
)

fig_sonia.show()


Fetching SONIA data from Bank of England Statistics API...
Period: 29 May 2025 → 29 May 2026

Records retrieved: 252
      Date  SONIA Rate (%)
2026-05-13          3.7290
2026-05-14          3.7300
2026-05-15          3.7299
2026-05-18          3.7297
2026-05-19          3.7302
2026-05-20          3.7301
2026-05-21          3.7309
2026-05-22          3.7301
2026-05-26          3.7290
2026-05-27          3.7290


In [5]:

# ── Combined chart: Base Rate vs SONIA ───────────────────────────────────────
fig_combined = go.Figure()

# Base Rate (step line — it changes discretely)
fig_combined.add_trace(go.Scatter(
    x=df["Date"],
    y=df["Base Rate (%)"],
    mode="lines+markers",
    line=dict(color="#C8102E", width=2.5, shape="hv"),  # step line
    marker=dict(size=8, color="#C8102E"),
    name="Base Rate",
    hovertemplate="<b>%{x|%d %b %Y}</b><br>Base Rate: <b>%{y:.2f}%</b><extra></extra>"
))

# SONIA (continuous daily line)
fig_combined.add_trace(go.Scatter(
    x=df_sonia["Date"],
    y=df_sonia["SONIA Rate (%)"],
    mode="lines",
    line=dict(color="#003087", width=1.5),
    name="SONIA",
    hovertemplate="<b>%{x|%d %b %Y}</b><br>SONIA: <b>%{y:.4f}%</b><extra></extra>"
))

# Compute a tight y range with padding
all_rates = pd.concat([df["Base Rate (%)"], df_sonia["SONIA Rate (%)"]])
y_min = all_rates.min()
y_max = all_rates.max()
y_pad = (y_max - y_min) * 0.5  # 50% padding above/below the spread

fig_combined.update_layout(
    title=dict(
        text="Bank of England Base Rate vs SONIA — Last 12 Months",
        font=dict(size=20),
        x=0.5
    ),
    xaxis=dict(title="Date", showgrid=True, gridcolor="#e5e5e5", tickformat="%b %Y"),
    yaxis=dict(
        title="Rate (%)",
        showgrid=True,
        gridcolor="#e5e5e5",
        ticksuffix="%",
        range=[y_min - y_pad, y_max + y_pad],  # tight range to show spread clearly
    ),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    plot_bgcolor="white",
    paper_bgcolor="white",
    hovermode="x unified",
    annotations=[dict(
        text='Source: <a href="https://www.bankofengland.co.uk/statistics">Bank of England Statistics</a>'
             ' (IUMBEDR, IUDSOIA)',
        showarrow=False, xref="paper", yref="paper",
        x=1, y=-0.16, xanchor="right", font=dict(size=11, color="grey")
    )],
    margin=dict(b=90)
)

fig_combined.show()


In [4]:

import requests
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime, timedelta
from io import StringIO

# BoE direct daily SONIA term rate series codes (from BoE database)
# IUDSOIA  = Overnight
# IUDVWKA  = 1 Week
# IUDVWLA  = 2 Weeks
# IUDVWMA  = 1 Month
# IUDVWNA  = 3 Months
# IUDVWOA  = 6 Months
# IUDVWPA  = 12 Months

tenors = {
    "Overnight":  "IUDSOIA",
    "1 Week":     "IUDVWKA",
    "2 Weeks":    "IUDVWLA",
    "1 Month":    "IUDVWMA",
    "3 Months":   "IUDVWNA",
    "6 Months":   "IUDVWOA",
    "12 Months":  "IUDVWPA",
}

end_date = datetime.today()
start_date = end_date - timedelta(days=365)

def fetch_boe(code, label):
    url = (
        "https://www.bankofengland.co.uk/boeapps/database/_iadb-FromShowColumns.asp"
        "?csv.x=yes"
        f"&Datefrom={start_date.strftime('%d/%b/%Y')}"
        f"&Dateto={end_date.strftime('%d/%b/%Y')}"
        f"&SeriesCodes={code}"
        "&CSVF=TN"
        "&UsingCodes=Y"
    )
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    resp.raise_for_status()
    df = pd.read_csv(StringIO(resp.text))
    df.columns = ["Date", label]
    df["Date"]  = pd.to_datetime(df["Date"], format="mixed", dayfirst=True)
    df[label]   = pd.to_numeric(df[label], errors="coerce")
    return df.dropna().sort_values("Date").reset_index(drop=True)

print(f"Fetching SONIA term rates — {start_date.strftime('%d %b %Y')} → {end_date.strftime('%d %b %Y')}\n")

df_term = None
for label, code in tenors.items():
    print(f"  {label:12s} ({code})...", end=" ")
    try:
        df_s = fetch_boe(code, label)
        print(f"{len(df_s)} records")
        df_term = df_s if df_term is None else pd.merge(df_term, df_s, on="Date", how="outer")
    except Exception as e:
        print(f"FAILED: {e}")

df_term = df_term.sort_values("Date").reset_index(drop=True)
available = [c for c in tenors.keys() if c in df_term.columns]
print(f"\nShape: {df_term.shape}")
print(df_term.tail(5).to_string(index=False))

# ── Plotly chart ──────────────────────────────────────────────────────────────
colors = ["#a8d1f5", "#7ab8f0", "#4d9fe8", "#2080d0", "#1060a8", "#003d80", "#001f40"]

fig_term = go.Figure()

for tenor, color in zip(tenors.keys(), colors):
    if tenor not in available:
        continue
    fig_term.add_trace(go.Scatter(
        x=df_term["Date"],
        y=df_term[tenor],
        mode="lines",
        name=tenor,
        line=dict(color=color, width=1.8),
        hovertemplate=f"<b>%{{x|%d %b %Y}}</b><br>{tenor}: <b>%{{y:.4f}}%</b><extra></extra>"
    ))

all_vals = df_term[available].stack().dropna()
y_min, y_max = all_vals.min(), all_vals.max()
y_pad = (y_max - y_min) * 0.3

fig_term.update_layout(
    title=dict(
        text="SONIA Term Rates — Last 12 Months",
        font=dict(size=20),
        x=0.5
    ),
    xaxis=dict(title="Date", showgrid=True, gridcolor="#e5e5e5", tickformat="%b %Y"),
    yaxis=dict(
        title="Rate (%)",
        showgrid=True,
        gridcolor="#e5e5e5",
        ticksuffix="%",
        range=[y_min - y_pad, y_max + y_pad],
    ),
    legend=dict(title="Tenor", orientation="v", x=1.01, y=1, xanchor="left"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    hovermode="x unified",
    annotations=[dict(
        text='Source: <a href="https://www.bankofengland.co.uk/markets/sonia-benchmark">'
             'Bank of England Statistics (IUDVWKA, IUDVWLA, IUDVWMA series)</a>',
        showarrow=False, xref="paper", yref="paper",
        x=1, y=-0.12, xanchor="right", font=dict(size=11, color="grey")
    )],
    margin=dict(r=130, b=80)
)

fig_term.show()


Fetching SONIA term rates — 29 May 2025 → 29 May 2026

  Overnight    (IUDSOIA)... 252 records
  1 Week       (IUDVWKA)... 0 records
  2 Weeks      (IUDVWLA)... 0 records
  1 Month      (IUDVWMA)... FAILED: Error tokenizing data. C error: Expected 1 fields in line 7, saw 2

  3 Months     (IUDVWNA)... FAILED: Error tokenizing data. C error: Expected 1 fields in line 7, saw 2

  6 Months     (IUDVWOA)... FAILED: Error tokenizing data. C error: Expected 1 fields in line 7, saw 2

  12 Months    (IUDVWPA)... FAILED: Error tokenizing data. C error: Expected 1 fields in line 7, saw 2


Shape: (252, 4)
      Date  Overnight  1 Week  2 Weeks
2026-05-20     3.7301     NaN      NaN
2026-05-21     3.7309     NaN      NaN
2026-05-22     3.7301     NaN      NaN
2026-05-26     3.7290     NaN      NaN
2026-05-27     3.7290     NaN      NaN
